# Stratégies de chunking pour le RAG

Ce notebook compare **trois** stratégies de découpage de texte (chunking) et leur
impact sur un RAG, du plus naïf au plus robuste :

1. **Taille fixe** — on coupe tous les N caractères, sans rien respecter (le baseline naïf)
2. **Par paragraphe** — on coupe sur les sauts de ligne, sans regrouper ni chevauchement
3. **Récursif avec chevauchement** — hiérarchie de séparateurs + regroupement + overlap

Les trois sont testées sur le même corpus Wikipédia. Le **récursif** est la
stratégie retenue pour les trois stacks RAG du projet.

## Pourquoi le chunking est important

Les modèles d'embedding ont une fenêtre d'entrée limitée (≈ 256–512 tokens). Des
chunks plus courts et ciblés produisent des embeddings plus précis ; mais des
chunks *trop* courts perdent le contexte. Il faut trouver le juste équilibre :

| Facteur | Trop petit | Trop grand |
|--------|-----------|----------|
| Précision | Élevée (mais bruitée) | Faible (diluée) |
| Rappel | Faible (fragments) | Élevé (mais hors sujet) |
| Contexte | Perdu aux frontières | Préservé |
| Qualité de l'embedding | Bonne | Saturée |

Le **chevauchement** (overlap) atténue la perte de contexte : la fin d'un chunk
est répétée au début du suivant.

## Configuration

In [ ]:
# Dépendances de ce notebook (chunking + graphiques)
%pip install -q matplotlib datasets numpy

In [ ]:
import sys
from pathlib import Path

# Trouve la racine du projet (dossier contenant src/), que Jupyter soit lancé
# depuis notebooks/ ou depuis la racine du repo.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "shared").is_dir()),
    Path.cwd().parent,
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np

## Chargement des données Wikipédia

Simple English Wikipedia (Hugging Face), limité à 100 articles.

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train[:100]")
print(f"{len(ds)} articles chargés")
print(f"Exemple : '{ds[0]['title']}' ({len(ds[0]['text'])} caractères)")

In [ ]:
text_lengths = [len(doc["text"]) for doc in ds]
print(f"Longueurs des articles : min={min(text_lengths)}, max={max(text_lengths)}, "
      f"moyenne={np.mean(text_lengths):.0f}, médiane={np.median(text_lengths):.0f}")

> **Interprétation.** Les articles sont très **hétérogènes** (de quelques dizaines à plusieurs dizaines de milliers de caractères). La **moyenne ≫ la médiane** → distribution **asymétrique à droite** : quelques très longs articles tirent la moyenne vers le haut. C'est pourquoi il faut **découper** : on ne peut pas embedder un article entier d'un bloc.

---
## Stratégie 1 : taille fixe (le baseline naïf)

La plus simple possible : couper tous les `size` caractères, sans tenir compte
des mots, phrases ou paragraphes. C'est le point de comparaison « naïf » classique.

In [ ]:
def chunk_fixed(text: str, size: int = 500) -> list[str]:
    """Découpe tous les `size` caractères, sans respecter mots ni phrases."""
    return [text[i:i + size] for i in range(0, len(text), size)]


fixed_chunks = []
for doc in ds:
    fixed_chunks.extend(chunk_fixed(doc["text"]))

fixed_lens = [len(c) for c in fixed_chunks]
print(f"Stratégie 1 (Taille fixe) : {len(fixed_chunks)} chunks")
print(f"  Longueurs : min={min(fixed_lens)}, max={max(fixed_lens)}, "
      f"moyenne={np.mean(fixed_lens):.0f}, médiane={np.median(fixed_lens):.0f}")
# La coupe entre deux chunks tombe en plein milieu du texte :
print(f"  Coupe : ...{fixed_chunks[0][-35:]!r}  |  {fixed_chunks[1][:35]!r}...")

> **Interprétation.** Toutes les longueurs valent **exactement 500** (sauf le dernier chunk de chaque article) → distribution **parfaitement uniforme**, et c'est la stratégie qui produit le **moins** de chunks. **Mais** regarde la coupe ci-dessus : elle tombe **en plein milieu d'un mot / d'une phrase**. Uniforme ≠ bon : on détruit le sens aux frontières. C'est le piège du découpage le plus naïf.

---
## Stratégie 2 : découpage par paragraphe (sans chevauchement)

Un cran au-dessus : on coupe sur les paragraphes (`\n\n`), et on re-coupe par mots
tout paragraphe qui dépasse `max_len`. On respecte les frontières de paragraphes,
mais on ne **regroupe pas** les petits, et **aucun chevauchement**.

In [ ]:
def chunk_simple(text: str, max_len: int = 500) -> list[str]:
    """Découpage par paragraphe (sans chevauchement, sans regroupement)."""
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    for p in paras:
        if len(p) <= max_len:
            chunks.append(p)
        else:
            words = p.split()
            curr = []
            for w in words:
                if len(" ".join(curr + [w])) <= max_len:
                    curr.append(w)
                else:
                    chunks.append(" ".join(curr))
                    curr = [w]
            if curr:
                chunks.append(" ".join(curr))
    return chunks


simple_chunks = []
for doc in ds:
    simple_chunks.extend(chunk_simple(doc["text"]))

simple_lens = [len(c) for c in simple_chunks]
print(f"Stratégie 2 (Paragraphe) : {len(simple_chunks)} chunks")
print(f"  Longueurs : min={min(simple_lens)}, max={max(simple_lens)}, "
      f"moyenne={np.mean(simple_lens):.0f}, médiane={np.median(simple_lens):.0f}")
fragments = sorted({c for c in simple_chunks if len(c) <= 5})
print(f"  Fragments (≤ 5 car.) : {fragments[:6]}")

> **Interprétation.** On respecte les paragraphes, mais comme on ne **regroupe pas** les petits, un mini-paragraphe (parfois **2 caractères !**) devient un chunk à lui seul → des **fragments inutiles** (voir la liste ci-dessus), du bruit pour l'embedding. `max_len` n'est qu'un **plafond**, pas un plancher. Et toujours **aucun chevauchement**.

---
## Stratégie 3 : découpage récursif avec chevauchement

Notre stratégie : une hiérarchie de séparateurs (`\n\n` > `\n` > `. ` > ` `), un
**regroupement** des segments jusqu'à `max_size`, et un **chevauchement** de
caractères entre chunks. C'est `recursive_chunk` de `src/shared/chunker.py`.

In [ ]:
from shared.chunker import recursive_chunk, Chunk

recursive_chunks: list[Chunk] = []
for doc in ds:
    recursive_chunks.extend(recursive_chunk(
        doc["text"], max_size=500, overlap=50,
        metadata={"title": doc["title"], "url": doc["url"]},
    ))

recursive_lens = [len(c.text) for c in recursive_chunks]
print(f"Stratégie 3 (Récursif + chevauchement) : {len(recursive_chunks)} chunks")
print(f"  Longueurs : min={min(recursive_lens)}, max={max(recursive_lens)}, "
      f"moyenne={np.mean(recursive_lens):.0f}, médiane={np.median(recursive_lens):.0f}")
print(f"  Métadonnées : {recursive_chunks[0].metadata}")

> **Interprétation.** **Moins** de chunks que le paragraphe, mais plus **pleins** (médiane proche de la cible) et **sans fragments** (min élevé) car les petits segments sont **regroupés**. Le `max` dépasse légèrement 500 : c'est normal, c'est le **chevauchement** (≤ `overlap` caractères repris du chunk précédent). Frontières respectées **et** contexte préservé.

### Démonstration du chevauchement

Vérifions que le chevauchement préserve le contexte aux frontières.

In [ ]:
demo_chunks = recursive_chunk(ds[0]["text"], max_size=300, overlap=80)

c1, c2 = demo_chunks[0].text, demo_chunks[1].text
print("CHUNK 1 (100 derniers caractères) :")
print(f"  ...{c1[-100:]}")
print("\nCHUNK 2 (100 premiers caractères) :")
print(f"  {c2[:100]}...")

for length in range(min(len(c1), len(c2)), 0, -1):
    if c1[-length:] == c2[:length]:
        print(f"\nChevauchement détecté : {length} caractères")
        break

> **Interprétation.** Le chunk 2 **commence par la fin du chunk 1** : 80 caractères repris (= `overlap=80`). Une phrase à cheval sur deux chunks reste **entière dans au moins l'un des deux** → l'information à la frontière n'est pas perdue.

---
## Comparaison

Comparons les trois stratégies.

In [ ]:
strategies = {
    "Taille fixe": {"count": len(fixed_chunks), "lengths": fixed_lens},
    "Paragraphe": {"count": len(simple_chunks), "lengths": simple_lens},
    "Récursif (overlap=50)": {"count": len(recursive_chunks), "lengths": recursive_lens},
}

print(f"{'Stratégie':<24}{'Chunks':>8}{'Min':>6}{'Max':>6}{'Moyenne':>9}{'Médiane':>9}")
print("-" * 62)
for name, info in strategies.items():
    lens = info["lengths"]
    print(f"{name:<24}{info['count']:>8}{min(lens):>6}{max(lens):>6}"
          f"{np.mean(lens):>9.1f}{np.median(lens):>9.1f}")

> **Comment lire ce tableau.** Trois signaux à croiser :
> - **Taille fixe** : longueurs toutes = 500 (médiane = moyenne), le moins de chunks → **uniforme mais brutal** (coupe le sens).
> - **Paragraphe** : moyenne basse + **min minuscule** → **fragmenté**.
> - **Récursif** : **médiane proche de la cible**, pas de fragments → **homogène et propre**. ✅
>
> Règle : une bonne stratégie a une **médiane proche de la taille visée** *et* respecte le sens (pas de coupe au milieu d'un mot, pas de fragments).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = ["#ef4444", "#3b82f6", "#22c55e"]  # rouge=fixe, bleu=paragraphe, vert=récursif

for ax, (name, info), color in zip(axes, strategies.items(), colors):
    ax.hist(info["lengths"], bins=30, color=color, alpha=0.85, edgecolor="white")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Longueur du chunk (caractères)")
    ax.axvline(np.mean(info["lengths"]), color="black", linestyle="--", label="moyenne")
    ax.legend()

axes[0].set_ylabel("Fréquence")
fig.suptitle("Distribution des longueurs de chunks par stratégie", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
names = list(strategies.keys())
counts = [strategies[n]["count"] for n in names]
bars = ax.bar(names, counts, color=colors, edgecolor="white")
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            str(count), ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Nombre de chunks")
ax.set_title("Nombre total de chunks par stratégie")
plt.tight_layout()
plt.show()

> **Lecture des graphes.**
> - **Histogrammes** : *Taille fixe* = un **pic unique à 500** (uniforme) ; *Paragraphe* = **étalé vers les petites** longueurs (fragments) ; *Récursif* = **concentré vers 400–500** (chunks pleins). Ligne noire = moyenne.
> - **Barres** : le nombre de chunks. « Moins » ou « plus » n'est pas le critère — c'est la **qualité sémantique** des chunks qui compte.

## Points clés à retenir

Du plus naïf au plus robuste (corpus de 100 articles, cible 500 caractères) :

1. **Taille fixe** — uniforme et peu de chunks, **mais coupe en plein milieu des mots/phrases**. À éviter : on détruit le sens aux frontières.
2. **Par paragraphe** — respecte les paragraphes, mais ne regroupe pas les petits → **fragments parasites** (jusqu'à 2 caractères) et **aucun chevauchement**.
3. **Récursif avec chevauchement** — respecte les frontières, **regroupe** en chunks pleins et homogènes, et **préserve le contexte** via le chevauchement. ✅

**On retient le récursif** comme unique stratégie pour les trois stacks RAG du
projet. *(Chiffres issus d'un run de 100 articles ; ils varient avec le corpus et les paramètres.)*